In [3]:
from sklearn.metrics import precision_score, recall_score, f1_score
import os
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "codes" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "codes" / "agents"))
import agent_scorer as scorer

source_dir = str(PROJECT_ROOT / "outputs" / "opt_prompt") + "/"
files = sorted(f for f in os.listdir(source_dir) if f.endswith(".txt"))
# print(files)
import numpy as np
def evaluate_in_chunks(golds, preds, scorer, chunk_size=30000):
    n_chunks  = (len(golds) // chunk_size) + 1
    precisions, recalls, f1s = [], [], []

    for i in range(n_chunks):
        start = i * chunk_size
        end = (i + 1) * chunk_size if i < n_chunks - 1 else len(golds)  # last chunk takes remainder

        g_chunk = golds[start:end]
        p_chunk = preds[start:end]

        if len(g_chunk) == 0:  # skip empty chunks
            continue

        sc = scorer.score(g_chunk, p_chunk, False)  # returns (P, R, F1)
        precisions.append(sc[0])
        recalls.append(sc[1])
        f1s.append(sc[2])

    precisions, recalls, f1s = np.array(precisions), np.array(recalls), np.array(f1s)

    # for p, r, f1 in zip(precisions, recalls, f1s):
    #     print(f"{p*100:.2f}\t{r*100:.2f}\t{f1*100:.2f}")

    results = {
        "mean_P": np.mean(precisions) * 100,
        "std_P": np.std(precisions) * 100,
        "mean_R": np.mean(recalls) * 100,
        "std_R": np.std(recalls) * 100,
        "mean_F1": np.mean(f1s) * 100,
        "std_F1": np.std(f1s) * 100,
    }

    return results

for f in files:
    print(f)

fewrel_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
fewrel_gemma_etgpo_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
fewrel_gemma_etgpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
fewrel_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
fewrel_gemma_evoprompt_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
fewrel_gemma_evoprompt_node_x_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
fewrel_gemma_evoprompt_node_x_greater-tg_google-gemma-3-

In [4]:
cccnt = 0
for ii, ff in enumerate(files):
    source = ff
    # if not ff.startswith('Qwen') and not ff.startswith("meta-llama") and not ff.startswith("microsoft"):
    # if not (ff.startswith('meta-llama') or "gemma-2-9b-it" in ff or ff.startswith("microsoft")) :
    # if not ("Qwen-Qwen3-4B-5shots_fs_tacred_test_episodes_5shots_softmatch.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20251202_hybrid.txt" in ff) :
    #     continue
    # if "main_test_qwen4b-ps_3-initial_prompt_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260429.txt" not in source:
    #     continue
    if (not source.endswith(".txt")):
        continue

    # if "tacred_qwen_rpo_node_y_gradpo" not in source:
    #     continue
    print(f"{ii} ************************** ")
    cccnt += 1
    print(source)
    p_list = []
    q_list = []
    rl_list = []
    r_list = []

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            p_list.append(line)

        if '@@@@@@@@@@' in line:
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            q_list.append(line)

        if line == '$$$$$ query relation':
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            rl_list.append(line)

        if line == '$$$$$ relation list':
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            r_list.append(line)

        if line == '$$$$$ votes':
            ready = True
        else:
            ready = False


    for i in range(len(rl_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(rl_list[i])
        rl_list[i] = reconstructed_list

    for i in range(len(r_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(r_list[i])
        r_list[i] = reconstructed_list


    for i in range(len(p_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(p_list[i])
        p_list[i] = reconstructed_list

    # print(p_list[-1][0][0]['content'])

    cnt = 0
    trans = []
    trans_ = []
    for i in range(len(rl_list)):
        dec = None
        dec_ = []
        mul = False

        # ens_res = ens[i]

        for wi, (rrr, yyy) in enumerate (zip(r_list[i], rl_list[i])):
            if 'yes' == rrr:

                # if enable_ens:
                #     widx = wi * 2
                #     sub_yes = ens_res[widx].lower() == "yes"
                #     obj_yes = ens_res[widx + 1].lower() == "yes"

                #     if not (sub_yes and obj_yes):
                #         continue

                if dec != None:
                    mul = True

                dec = yyy
                dec_.append(yyy)

        if dec == None:
            dec = 'no_relation'
        elif mul == True:
            dec = '#multiple#'
        else:
            dec = dec

        trans.append(dec)
        trans_.append(dec_)

    # print(cnt)

    temp = r_list
    r_list = trans
    trans = temp

    golds = []
    preds = []

    for i in range(len(q_list)):
        q = q_list[i]
        rl = rl_list[i]
        r = r_list[i]

        if q in rl:
            assert q != 'no_relation'
            golds.append(q)
        else:
            golds.append('no_relation')

        preds.append(r)


    res = evaluate_in_chunks(golds, preds, scorer, chunk_size=30000)
    print(len(golds))
    print(f"{(res['mean_P']):04.1f} ± {res['std_P']:4.2f}\t{(res['mean_R']):04.1f} ± {res['std_R']:4.2f}\t{(res['mean_F1']):04.1f} ± {res['std_F1']:4.2f}")
    print(f"& {(res['mean_P']):04.1f} & {res['std_P']:4.2f} & {(res['mean_R']):04.1f} & {res['std_R']:4.2f} & {(res['mean_F1']):04.1f} & {res['std_F1']:4.2f}")
    # sc = scorer.score(golds, preds, False)
    # print(f"{sc[0]*100:.2f}\t{sc[1]*100:.2f}\t{sc[2]*100:.2f}")
  

0 ************************** 
fewrel_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
33.8 ± 1.50	36.6 ± 2.14	35.2 ± 1.73
& 33.8 & 1.50 & 36.6 & 2.14 & 35.2 & 1.73
1 ************************** 
fewrel_gemma_etgpo_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
35.8 ± 1.51	34.2 ± 2.08	35.0 ± 1.76
& 35.8 & 1.51 & 34.2 & 2.08 & 35.0 & 1.76
2 ************************** 
fewrel_gemma_etgpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
35.9 ± 1.96	22.7 ± 1.30	27.8 ± 1.52
& 35.9 & 1.96 & 22.7 & 1.30 & 27.8 & 1.52
3 ************************** 
fewrel_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
42.

In [ ]:
0 ************************** 
fewrel_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
33.8 ± 1.50	36.6 ± 2.14	35.2 ± 1.73
& 33.8 & 1.50 & 36.6 & 2.14 & 35.2 & 1.73
1 ************************** 
fewrel_gemma_etgpo_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
35.8 ± 1.51	34.2 ± 2.08	35.0 ± 1.76
& 35.8 & 1.51 & 34.2 & 2.08 & 35.0 & 1.76
2 ************************** 
fewrel_gemma_etgpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
35.9 ± 1.96	22.7 ± 1.30	27.8 ± 1.52
& 35.9 & 1.96 & 22.7 & 1.30 & 27.8 & 1.52
3 ************************** 
fewrel_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
42.8 ± 1.54	20.7 ± 1.57	27.9 ± 1.67
& 42.8 & 1.54 & 20.7 & 1.57 & 27.9 & 1.67
4 ************************** 
fewrel_gemma_evoprompt_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
37.3 ± 1.43	30.7 ± 1.91	33.7 ± 1.71
& 37.3 & 1.43 & 30.7 & 1.91 & 33.7 & 1.71
5 ************************** 
fewrel_gemma_evoprompt_node_x_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
40.2 ± 1.29	26.5 ± 1.69	32.0 ± 1.61
& 40.2 & 1.29 & 26.5 & 1.69 & 32.0 & 1.61
6 ************************** 
fewrel_gemma_evoprompt_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
42.4 ± 1.23	21.3 ± 1.58	28.3 ± 1.62
& 42.4 & 1.23 & 21.3 & 1.58 & 28.3 & 1.62
7 ************************** 
fewrel_gemma_evoprompt_node_x_greater_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
38.7 ± 1.47	28.4 ± 2.05	32.8 ± 1.86
& 38.7 & 1.47 & 28.4 & 2.05 & 32.8 & 1.86
8 ************************** 
fewrel_gemma_evoprompt_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
38.8 ± 1.37	25.6 ± 1.89	30.9 ± 1.79
& 38.8 & 1.37 & 25.6 & 1.89 & 30.9 & 1.79
9 ************************** 
fewrel_gemma_evoprompt_node_y_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
37.2 ± 1.33	33.3 ± 1.89	35.2 ± 1.64
& 37.2 & 1.33 & 33.3 & 1.89 & 35.2 & 1.64
10 ************************** 
fewrel_gemma_evoprompt_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
35.7 ± 1.09	39.5 ± 1.73	37.5 ± 1.33
& 35.7 & 1.09 & 39.5 & 1.73 & 37.5 & 1.33 done
11 ************************** 
fewrel_gemma_evoprompt_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
37.4 ± 1.28	33.2 ± 1.70	35.2 ± 1.51
& 37.4 & 1.28 & 33.2 & 1.70 & 35.2 & 1.51 done
12 ************************** 
fewrel_gemma_evoprompt_node_y_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
35.7 ± 0.85	37.9 ± 1.54	36.8 ± 1.17
& 35.7 & 0.85 & 37.9 & 1.54 & 36.8 & 1.17 done
13 ************************** 
fewrel_gemma_initial_prompt_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
39.9 ± 2.13	18.6 ± 1.12	25.3 ± 1.27
& 39.9 & 2.13 & 18.6 & 1.12 & 25.3 & 1.27 done
14 ************************** 
fewrel_gemma_rpo_node_x_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
35.1 ± 1.06	26.7 ± 1.27	30.3 ± 1.14
& 35.1 & 1.06 & 26.7 & 1.27 & 30.3 & 1.14
15 ************************** 
fewrel_gemma_rpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
30.2 ± 0.85	35.7 ± 1.70	32.7 ± 1.12
& 30.2 & 0.85 & 35.7 & 1.70 & 32.7 & 1.12
16 ************************** 
fewrel_gemma_rpo_node_x_greater_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
35.0 ± 1.13	26.5 ± 1.35	30.2 ± 1.23
& 35.0 & 1.13 & 26.5 & 1.35 & 30.2 & 1.23
17 ************************** 
fewrel_gemma_rpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
32.8 ± 1.11	43.8 ± 2.03	37.5 ± 1.30
& 32.8 & 1.11 & 43.8 & 2.03 & 37.5 & 1.30 done
18 ************************** 
fewrel_gemma_rpo_node_y_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
34.8 ± 1.39	29.9 ± 1.42	32.2 ± 1.30
& 34.8 & 1.39 & 29.9 & 1.42 & 32.2 & 1.30 done
19 ************************** 
fewrel_gemma_rpo_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
32.5 ± 1.28	34.1 ± 1.77	33.3 ± 1.40
& 32.5 & 1.28 & 34.1 & 1.77 & 33.3 & 1.40 done
20 ************************** 
fewrel_gemma_rpo_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
35.2 ± 1.19	29.0 ± 1.54	31.8 ± 1.28
& 35.2 & 1.19 & 29.0 & 1.54 & 31.8 & 1.28 done
21 ************************** 
fewrel_gemma_rpo_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
34.6 ± 1.20	31.1 ± 1.47	32.7 ± 1.25
& 34.6 & 1.20 & 31.1 & 1.47 & 32.7 & 1.25
22 ************************** 
fewrel_qwen_etgpo_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
35.6 ± 1.35	34.5 ± 0.93	35.0 ± 1.06
& 35.6 & 1.35 & 34.5 & 0.93 & 35.0 & 1.06
23 ************************** 
fewrel_qwen_etgpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
34.3 ± 1.22	36.0 ± 0.95	35.1 ± 1.04
& 34.3 & 1.22 & 36.0 & 0.95 & 35.1 & 1.04
24 ************************** 
fewrel_qwen_etgpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
33.8 ± 1.14	36.8 ± 0.81	35.2 ± 0.93
& 33.8 & 1.14 & 36.8 & 0.81 & 35.2 & 0.93
25 ************************** 
fewrel_qwen_etgpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
36.2 ± 1.44	35.1 ± 1.21	35.7 ± 1.24
& 36.2 & 1.44 & 35.1 & 1.21 & 35.7 & 1.24
26 ************************** 
fewrel_qwen_evoprompt_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
35.5 ± 0.71	36.3 ± 0.67	35.9 ± 0.59
& 35.5 & 0.71 & 36.3 & 0.67 & 35.9 & 0.59
27 ************************** 
fewrel_qwen_evoprompt_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
33.5 ± 1.05	40.6 ± 0.51	36.7 ± 0.79
& 33.5 & 1.05 & 40.6 & 0.51 & 36.7 & 0.79
28 ************************** 
fewrel_qwen_evoprompt_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
35.7 ± 1.11	36.2 ± 0.78	35.9 ± 0.85
& 35.7 & 1.11 & 36.2 & 0.78 & 35.9 & 0.85
29 ************************** 
fewrel_qwen_evoprompt_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
35.5 ± 0.55	37.0 ± 0.80	36.2 ± 0.50
& 35.5 & 0.55 & 37.0 & 0.80 & 36.2 & 0.50
30 ************************** 
fewrel_qwen_evoprompt_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
35.8 ± 0.68	40.6 ± 0.48	38.0 ± 0.56
& 35.8 & 0.68 & 40.6 & 0.48 & 38.0 & 0.56
31 ************************** 
fewrel_qwen_initial_prompt_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
29.9 ± 0.72	44.1 ± 0.76	35.6 ± 0.66
& 29.9 & 0.72 & 44.1 & 0.76 & 35.6 & 0.66
32 ************************** 
fewrel_qwen_rpo_node_x_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
42.4 ± 0.97	29.5 ± 0.55	34.8 ± 0.60
& 42.4 & 0.97 & 29.5 & 0.55 & 34.8 & 0.60
33 ************************** 
fewrel_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
42.9 ± 1.27	30.7 ± 0.64	35.8 ± 0.80
& 42.9 & 1.27 & 30.7 & 0.64 & 35.8 & 0.80
34 ************************** 
fewrel_qwen_rpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
42.2 ± 1.32	29.1 ± 0.88	34.5 ± 0.99
& 42.2 & 1.32 & 29.1 & 0.88 & 34.5 & 0.99
35 ************************** 
fewrel_qwen_rpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
43.2 ± 0.91	29.9 ± 0.79	35.3 ± 0.82
& 43.2 & 0.91 & 29.9 & 0.79 & 35.3 & 0.82
36 ************************** 
fewrel_qwen_rpo_node_y_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
33.5 ± 0.98	39.6 ± 0.66	36.3 ± 0.74
& 33.5 & 0.98 & 39.6 & 0.66 & 36.3 & 0.74
37 ************************** 
fewrel_qwen_rpo_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
34.2 ± 0.94	40.2 ± 0.94	36.9 ± 0.82
& 34.2 & 0.94 & 40.2 & 0.94 & 36.9 & 0.82
38 ************************** 
fewrel_qwen_rpo_node_y_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
33.1 ± 1.03	40.8 ± 0.48	36.6 ± 0.76
& 33.1 & 1.03 & 40.8 & 0.48 & 36.6 & 0.76
39 ************************** 
fewrel_qwen_rpo_node_y_greater-tg_Qwen-Qwen3-4B-1shots_fs_fewrel_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
32.8 ± 0.93	40.9 ± 0.70	36.4 ± 0.74
& 32.8 & 0.93 & 40.9 & 0.70 & 36.4 & 0.74
40 ************************** 
tacred_gemma_etgpo_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
12.1 ± 0.43	14.4 ± 0.36	13.1 ± 0.32
& 12.1 & 0.43 & 14.4 & 0.36 & 13.1 & 0.32
41 ************************** 
tacred_gemma_etgpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
11.5 ± 0.51	18.2 ± 0.77	14.1 ± 0.58
& 11.5 & 0.51 & 18.2 & 0.77 & 14.1 & 0.58
42 ************************** 
tacred_gemma_etgpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
11.0 ± 0.40	14.4 ± 0.62	12.5 ± 0.41
& 11.0 & 0.40 & 14.4 & 0.62 & 12.5 & 0.41
43 ************************** 
tacred_gemma_evoprompt_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
11.7 ± 0.55	19.3 ± 0.55	14.5 ± 0.53
& 11.7 & 0.55 & 19.3 & 0.55 & 14.5 & 0.53
44 ************************** 
tacred_gemma_evoprompt_node_x_gradpo-prob_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
11.7 ± 0.68	19.1 ± 0.80	14.5 ± 0.72
& 11.7 & 0.68 & 19.1 & 0.80 & 14.5 & 0.72
45 ************************** 
tacred_gemma_evoprompt_node_x_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
11.4 ± 0.71	17.9 ± 0.81	13.9 ± 0.72
& 11.4 & 0.71 & 17.9 & 0.81 & 13.9 & 0.72
46 ************************** 
tacred_gemma_evoprompt_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
13.7 ± 0.70	12.7 ± 0.20	13.1 ± 0.27
& 13.7 & 0.70 & 12.7 & 0.20 & 13.1 & 0.27
47 ************************** 
tacred_gemma_evoprompt_node_y_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
12.5 ± 0.54	08.5 ± 0.48	10.1 ± 0.42
& 12.5 & 0.54 & 08.5 & 0.48 & 10.1 & 0.42
48 ************************** 
tacred_gemma_evoprompt_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
12.2 ± 0.77	08.9 ± 0.44	10.3 ± 0.50
& 12.2 & 0.77 & 08.9 & 0.44 & 10.3 & 0.50
49 ************************** 
tacred_gemma_evoprompt_node_y_gradpo-prob_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
12.4 ± 0.58	08.3 ± 0.51	09.9 ± 0.46
& 12.4 & 0.58 & 08.3 & 0.51 & 09.9 & 0.46
50 ************************** 
tacred_gemma_evoprompt_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
12.6 ± 0.59	09.0 ± 0.49	10.5 ± 0.39
& 12.6 & 0.59 & 09.0 & 0.49 & 10.5 & 0.39
51 ************************** 
tacred_gemma_evoprompt_node_y_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
12.3 ± 0.70	08.7 ± 0.39	10.2 ± 0.39
& 12.3 & 0.70 & 08.7 & 0.39 & 10.2 & 0.39
52 ************************** 
tacred_gemma_evoprompt_node_y_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
11.4 ± 0.74	09.6 ± 0.57	10.4 ± 0.59
& 11.4 & 0.74 & 09.6 & 0.57 & 10.4 & 0.59
53 ************************** 
tacred_gemma_initial_prompt_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
10.4 ± 0.56	12.5 ± 0.65	11.3 ± 0.53
& 10.4 & 0.56 & 12.5 & 0.65 & 11.3 & 0.53
54 ************************** 
tacred_gemma_rpo_node_x_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
15.8 ± 1.12	12.4 ± 0.83	13.9 ± 0.94
& 15.8 & 1.12 & 12.4 & 0.83 & 13.9 & 0.94
55 ************************** 
tacred_gemma_rpo_node_x_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
15.9 ± 1.25	12.4 ± 0.95	13.9 ± 1.06
& 15.9 & 1.25 & 12.4 & 0.95 & 13.9 & 1.06
56 ************************** 
tacred_gemma_rpo_node_x_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
14.6 ± 1.11	13.5 ± 0.96	14.0 ± 1.01
& 14.6 & 1.11 & 13.5 & 0.96 & 14.0 & 1.01
57 ************************** 
tacred_gemma_rpo_node_x_lpo_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
15.1 ± 0.63	15.0 ± 0.49	15.0 ± 0.54
& 15.1 & 0.63 & 15.0 & 0.49 & 15.0 & 0.54
58 ************************** 
tacred_gemma_rpo_node_y_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
09.8 ± 0.33	14.4 ± 0.72	11.7 ± 0.45
& 09.8 & 0.33 & 14.4 & 0.72 & 11.7 & 0.45
59 ************************** 
tacred_gemma_rpo_node_y_gradpo-gen_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
10.2 ± 0.20	14.3 ± 0.31	11.9 ± 0.11
& 10.2 & 0.20 & 14.3 & 0.31 & 11.9 & 0.11
60 ************************** 
tacred_gemma_rpo_node_y_greater-tg_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
09.8 ± 0.27	15.6 ± 0.79	12.0 ± 0.43
& 09.8 & 0.27 & 15.6 & 0.79 & 12.0 & 0.43
61 ************************** 
tacred_gemma_rpo_node_y_greater_google-gemma-3-4b-it-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260523.txt
150000
09.6 ± 0.31	12.5 ± 0.31	10.9 ± 0.27
& 09.6 & 0.31 & 12.5 & 0.31 & 10.9 & 0.27
62 ************************** 
tacred_qwen_etgpo_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
21.5 ± 0.54	41.4 ± 1.80	28.3 ± 0.75
& 21.5 & 0.54 & 41.4 & 1.80 & 28.3 & 0.75
63 ************************** 
tacred_qwen_etgpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
22.0 ± 0.49	37.3 ± 1.55	27.7 ± 0.66
& 22.0 & 0.49 & 37.3 & 1.55 & 27.7 & 0.66
64 ************************** 
tacred_qwen_etgpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
24.1 ± 1.17	29.0 ± 0.91	26.3 ± 1.03
& 24.1 & 1.17 & 29.0 & 0.91 & 26.3 & 1.03
65 ************************** 
tacred_qwen_etgpo_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
27.5 ± 0.87	28.1 ± 1.27	27.8 ± 0.92
& 27.5 & 0.87 & 28.1 & 1.27 & 27.8 & 0.92
66 ************************** 
tacred_qwen_etgpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
21.2 ± 0.56	43.2 ± 1.32	28.4 ± 0.67
& 21.2 & 0.56 & 43.2 & 1.32 & 28.4 & 0.67
67 ************************** 
tacred_qwen_etgpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
21.3 ± 0.73	42.3 ± 1.56	28.3 ± 0.89
& 21.3 & 0.73 & 42.3 & 1.56 & 28.3 & 0.89
68 ************************** 
tacred_qwen_evoprompt_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
19.4 ± 0.80	33.7 ± 2.24	24.6 ± 1.21
& 19.4 & 0.80 & 33.7 & 2.24 & 24.6 & 1.21
69 ************************** 
tacred_qwen_evoprompt_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
20.7 ± 0.83	31.1 ± 1.49	24.8 ± 0.99
& 20.7 & 0.83 & 31.1 & 1.49 & 24.8 & 0.99
70 ************************** 
tacred_qwen_evoprompt_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
19.9 ± 0.79	33.0 ± 1.86	24.8 ± 1.08
& 19.9 & 0.79 & 33.0 & 1.86 & 24.8 & 1.08
71 ************************** 
tacred_qwen_evoprompt_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
20.3 ± 1.06	29.6 ± 1.90	24.1 ± 1.31
& 20.3 & 1.06 & 29.6 & 1.90 & 24.1 & 1.31
72 ************************** 
tacred_qwen_evoprompt_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
20.0 ± 0.72	30.5 ± 1.81	24.2 ± 1.04
& 20.0 & 0.72 & 30.5 & 1.81 & 24.2 & 1.04
73 ************************** 
tacred_qwen_evoprompt_node_y_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
21.6 ± 0.53	28.4 ± 1.77	24.5 ± 0.98
& 21.6 & 0.53 & 28.4 & 1.77 & 24.5 & 0.98
74 ************************** 
tacred_qwen_evoprompt_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
18.5 ± 0.54	32.1 ± 1.56	23.4 ± 0.74
& 18.5 & 0.54 & 32.1 & 1.56 & 23.4 & 0.74
75 ************************** 
tacred_qwen_evoprompt_node_y_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
22.1 ± 0.38	28.2 ± 1.44	24.7 ± 0.78
& 22.1 & 0.38 & 28.2 & 1.44 & 24.7 & 0.78
76 ************************** 
tacred_qwen_evoprompt_node_y_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
21.0 ± 0.95	29.2 ± 1.93	24.4 ± 1.26
& 21.0 & 0.95 & 29.2 & 1.93 & 24.4 & 1.26
77 ************************** 
tacred_qwen_initial_prompt_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
17.6 ± 0.44	38.2 ± 1.38	24.0 ± 0.59
& 17.6 & 0.44 & 38.2 & 1.38 & 24.0 & 0.59
78 ************************** 
tacred_qwen_rpo_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
30.8 ± 1.12	23.3 ± 1.34	26.5 ± 1.19
& 30.8 & 1.12 & 23.3 & 1.34 & 26.5 & 1.19
79 ************************** 
tacred_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
29.5 ± 1.25	28.5 ± 1.66	29.0 ± 1.37
& 29.5 & 1.25 & 28.5 & 1.66 & 29.0 & 1.37
80 ************************** 
tacred_qwen_rpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
29.9 ± 1.04	25.3 ± 1.05	27.4 ± 0.94
& 29.9 & 1.04 & 25.3 & 1.05 & 27.4 & 0.94
81 ************************** 
tacred_qwen_rpo_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
27.9 ± 1.58	26.5 ± 1.69	27.2 ± 1.53
& 27.9 & 1.58 & 26.5 & 1.69 & 27.2 & 1.53
82 ************************** 
tacred_qwen_rpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
32.1 ± 1.44	21.8 ± 1.58	25.9 ± 1.43
& 32.1 & 1.44 & 21.8 & 1.58 & 25.9 & 1.43
83 ************************** 
tacred_qwen_rpo_node_x_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
30.4 ± 1.41	21.0 ± 0.90	24.8 ± 0.98
& 30.4 & 1.41 & 21.0 & 0.90 & 24.8 & 0.98
84 ************************** 
tacred_qwen_rpo_node_y_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
35.8 ± 1.19	23.4 ± 1.26	28.3 ± 1.13
& 35.8 & 1.19 & 23.4 & 1.26 & 28.3 & 1.13
85 ************************** 
tacred_qwen_rpo_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
35.3 ± 1.36	24.5 ± 1.53	28.9 ± 1.46
& 35.3 & 1.36 & 24.5 & 1.53 & 28.9 & 1.46
86 ************************** 
tacred_qwen_rpo_node_y_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
37.9 ± 1.24	22.7 ± 0.73	28.4 ± 0.68
& 37.9 & 1.24 & 22.7 & 0.73 & 28.4 & 0.68
87 ************************** 
tacred_qwen_rpo_node_y_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
37.7 ± 1.76	21.0 ± 1.15	26.9 ± 1.21
& 37.7 & 1.76 & 21.0 & 1.15 & 26.9 & 1.21
88 ************************** 
tacred_qwen_rpo_node_y_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
35.8 ± 1.18	24.1 ± 1.57	28.8 ± 1.38
& 35.8 & 1.18 & 24.1 & 1.57 & 28.8 & 1.38
89 ************************** 
tacred_qwen_rpo_node_y_lpo_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260522.txt
150000
35.3 ± 1.71	29.1 ± 1.72	31.9 ± 1.65
& 35.3 & 1.71 & 29.1 & 1.72 & 31.9 & 1.65